In [1]:
from dotenv import load_dotenv
load_dotenv()


True

# Baseline

In [2]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=1024, load_in_4bit=False)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/Qwen3-4B-Instruct-2507 does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
FastLanguageModel.for_inference(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layerno

In [4]:
def generate_response(messages, model, tokenizer):
    # generate response
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    responses = tokenizer.decode(outputs[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return responses


msg = [{'role': 'user', 'content': 'say hello'}]
messages = [msg, msg]
print(generate_response(messages, model, tokenizer))


--- Logging error ---
Traceback (most recent call last):
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/home/lab/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", lin

['Hello! 🌟 How can I assist you today? 😊', 'Hello! 🌟 How can I assist you today? 😊']


## IF-Eval

In [5]:
from instruction_following_eval import get_examples, evaluate_instruction_following
from tqdm import tqdm

In [30]:
examples = get_examples()
examples[0]

{'key': 1000,
 'prompt': 'Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*.',
 'instruction_id_list': ['punctuation:no_comma',
  'detectable_format:number_highlighted_sections',
  'length_constraints:number_words'],
 'kwargs': [{},
  {'num_highlights': 3},
  {'relation': 'at least', 'num_words': 300}]}

In [55]:
BATCH_SIZE = 16
responses = []
for i in tqdm(range(0, len(examples), BATCH_SIZE)):
    batch = examples[i:i+BATCH_SIZE]
    _msgs = [[{'role': 'user', 'content': example['prompt']}] for example in batch]
    responses.extend(generate_response(_msgs, model, tokenizer))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [13:12<00:00, 23.31s/it]


In [57]:
metrics = evaluate_instruction_following(examples, responses)
metrics


{'prompt_level_strict_accuracy': 0.822550831792976,
 'inst_level_strict_accuracy': 0.8752997601918465,
 'prompt_level_loose_accuracy': 0.8558225508317929,
 'inst_level_loose_accuracy': 0.8968824940047961}

## Internal QA Eval

The way we are doing this is just using LLM-as judge and binary scoring of correct answer or incorrect.

In [6]:
qa_examples = [dict(question="What is the capital of France?", golden_answer="Paris"),]
response = generate_response([{'role': 'user', 'content': qa_examples[0]['question']}], model, tokenizer)[0]
response

'The capital of France is Paris.'

In [7]:
from openai import AsyncOpenAI
import os

client = AsyncOpenAI(api_key=os.getenv("TOGETHER_API_KEY"), base_url="https://api.together.xyz/v1")

In [8]:
from pydantic import BaseModel

class JudgeResponse(BaseModel):
    gold_facts: list[str]
    response_facts: list[str]
    is_correct: bool


In [10]:
import json

judge_prompt = """
You are a judge for a question-answering task.

You are given a question, a golden answer and a response.

You need to determine if the response is correct or incorrect.

You will be given a score of 1 if the response is correct and 0 if it is incorrect.
"""

async def judge_response(question, golden_answer, response):    
    judge_response = await client.chat.completions.create(
        model="zai-org/GLM-4.7",
        messages=[{"role": "system", "content": judge_prompt}, {"role": "user", "content": f"Question: {question}\nGolden Answer: {golden_answer}\nResponse: {response}"}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "judge_response",
                "schema": JudgeResponse.model_json_schema(),
            },
        },
    )
    return json.loads(judge_response.choices[0].message.content)['is_correct']


print(await judge_response(qa_examples[0]['question'], qa_examples[0]['golden_answer'], response))

True


In [12]:
test_data = []
with open('knowledge-ingestion-test/test.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip(): test_data.append(json.loads(line.strip()))

test_data[0]

{'messages': [{'role': 'user',
   'content': 'How does the Geiger counter behave while Eddie is using it in his dream at Cedar Point?'},
  {'role': 'assistant', 'content': 'The Geiger counter clicks rapidly.'}]}

In [ ]:
model_responses = []
BATCH_SIZE = 16
for i in tqdm(range(0, len(test_data), BATCH_SIZE)):
    batch = test_data[i:i+BATCH_SIZE]
    _msgs = [example['messages'][:-1] for example in batch]
    model_responses.extend(generate_response(_msgs, model, tokenizer))

model_responses[0]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [01:27<00:00, 12.56s/it]


"There is no known instance or canonical story in which a Geiger counter is used by a character named Eddie at Cedar Point in a dream, nor is there a widely recognized reference to such a scene in literature, film, or popular culture.\n\nTherefore, based on available information, the Geiger counter does not behave in any documented or established way during such a dream scenario involving Eddie at Cedar Point.\n\nIf this question is referencing a fictional or speculative scenario (such as a creative writing piece, a dream sequence in a movie, or a fan-made story), the behavior of the Geiger counter would depend entirely on the author's intent — for example, it might detect radiation as a metaphor for anxiety, or it might remain silent to symbolize safety or normalcy.\n\nIn summary:  \n**In reality or established media, the Geiger counter does not behave in any specific way during Eddie’s dream at Cedar Point — because such a scene does not exist.**  \n\nIf you have a specific story, bo

In [17]:
# evaluation
import asyncio

results = []
for i in tqdm(range(0, len(test_data), BATCH_SIZE)):
    tasks = []
    for item, response in zip(test_data[i:i+BATCH_SIZE], model_responses[i:i+BATCH_SIZE]):
        q = item['messages'][0]['content']
        a = item['messages'][1]['content']
        tasks.append(judge_response(q, a, response))
    results.extend(await asyncio.gather(*tasks, return_exceptions=True))

sum(results) / len(results)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [04:39<00:00, 39.95s/it]


TypeError: unsupported operand type(s) for +: 'int' and 'JSONDecodeError'

In [ ]:
n_correct = 0
for x in results:
    if isinstance(x, bool) and x:
        n_correct += 1
n_correct / len(results)


Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Unterminated string starting at: line 1 column 288 (char 287)
Expecting ',' delimiter: line 3 column 43 (char 62)
Expecting value: line 1 column 1 (char 0)
Unterminated string starting at: line 1 column 202 (char 201)
